In [1]:
!pip -q install pandas nltk scikit-learn transformers torch

In [2]:
import pandas as pd
import nltk
import re

from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import pipeline

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.shape)
print(df["sentiment"].value_counts())

Saving IMDB Dataset.csv to IMDB Dataset.csv
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
# Clean text

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_review"] = df["review"].apply(clean_text)

print(df[["review", "clean_review"]].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  one of the other reviewers has mentioned that ...  
1  a wonderful little production the filming tech...  
2  i thought this was a wonderful way to spend ti...  
3  basically there s a family where a little boy ...  
4  petter mattei s love in the time of money is a...  


In [5]:

# Tokenization

df["tokens"] = df["clean_review"].apply(word_tokenize)

print(df[["clean_review", "tokens"]].head())

                                        clean_review  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production the filming tech...   
2  i thought this was a wonderful way to spend ti...   
3  basically there s a family where a little boy ...   
4  petter mattei s love in the time of money is a...   

                                              tokens  
0  [one, of, the, other, reviewers, has, mentione...  
1  [a, wonderful, little, production, the, filmin...  
2  [i, thought, this, was, a, wonderful, way, to,...  
3  [basically, there, s, a, family, where, a, lit...  
4  [petter, mattei, s, love, in, the, time, of, m...  


In [6]:
# Split data

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"],
    df["sentiment"],
    test_size=0.25,
    random_state=42,
    stratify=df["sentiment"]
)

print("Training:", len(X_train))
print("Testing:", len(X_test))

Training: 37500
Testing: 12500


In [7]:
# TF-IDF

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training:", X_train_tfidf.shape)
print("Testing:", X_test_tfidf.shape)

Training: (37500, 5000)
Testing: (12500, 5000)


In [8]:
# Logistic Regression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)

print("Model trained")

Model trained


In [9]:
# Accuracy

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
print(classification_report(y_test, predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.8928
              precision    recall  f1-score   support

    negative       0.90      0.89      0.89      6250
    positive       0.89      0.90      0.89      6250

    accuracy                           0.89     12500
   macro avg       0.89      0.89      0.89     12500
weighted avg       0.89      0.89      0.89     12500

Confusion Matrix:
[[5539  711]
 [ 629 5621]]


In [10]:
# Custom review

user_text = "This movie was amazing and I really enjoyed it"

cleaned_text = clean_text(user_text)

user_vector = vectorizer.transform([cleaned_text])

prediction = model.predict(user_vector)[0]

print("Review:", user_text)
print("Sentiment:", prediction)


Review: This movie was amazing and I really enjoyed it
Sentiment: positive


In [11]:

# Transformer

transformer = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

transformer_result = transformer(user_text)

print("Review:", user_text)
print("Sentiment:", transformer_result[0]["label"])
print("Confidence:", transformer_result[0]["score"])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Review: This movie was amazing and I really enjoyed it
Sentiment: POSITIVE
Confidence: 0.9998840093612671
